In [2]:
import os
from dotenv import load_dotenv,find_dotenv
load_dotenv()

True

In [3]:
api_key = os.getenv('OPENAI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')
WEATHER_API_KEY = os.getenv('WEATHER_API_KEY')

Tool calling for LLMs with native support for tool or function calling

In [4]:
from langchain_openai import ChatOpenAI

chatgpt = ChatOpenAI(model="gpt-4o", temperature=0)

In last few session we have created below 3 tools:

.multiply
.search_web_extract_info
.get_weather

Multiple Tool

In [5]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class Input_schema(BaseModel):
    a: float = Field(description="first number")
    b: float = Field(description="second number")


def multiply(a: float, b: float) -> float:
    """Use this tool to multiply two numbers."""
    return a * b

# Notice we have not used @tool decorator so we need to add gunction name, desc, ect.
multiply = StructuredTool.from_function(
    func=multiply,
    name="multiply",
    description="Use this tool to multiply two numbers.",
    args_schema=Input_schema,
    return_direct=True
    )

In [6]:
multiply.invoke({"a": 2, "b": 3})

6.0

In [7]:
from markitdown import MarkItDown
from langchain_community.tools.tavily_search import TavilySearchResults
from tqdm import tqdm
import requests
from langchain_core.tools import tool

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_63060/2822581896.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [8]:
tavily_tool = TavilySearchResults(max_results=5,
                                  search_depth='advanced',
                                  include_answer=False,
                                  include_raw_content=True)

/var/folders/y4/j6_cks6j2fb8p7zcm0hyft6r0000gn/T/ipykernel_63060/113296119.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5,


In [10]:
md = MarkItDown()

In [11]:

@tool
def search_web_extract_info(query: str) -> list:
    """Search the web for a query and extract useful information from search results."""

    results = tavily_tool.invoke(query)

    docs = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/142.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    }

    for result in tqdm(results):

        url = result["url"]
        print(f"Processing URL: {url}")
        try:

            response = requests.get(
                url,
                headers=headers,
                timeout=15,
                allow_redirects=True
            )

            response.raise_for_status()

            extracted_info = md.convert_response(response)

            title = extracted_info.title or ""
            content = extracted_info.text_content or ""

            if content.strip():
                docs.append(
                    f"{title.strip()}\n{content.strip()}"
                )

        except requests.exceptions.HTTPError as e:

            print(f"HTTP error: {url} -> {e}")

        except requests.exceptions.RequestException as e:

            print(f"Request failed: {url} -> {e}")

        except Exception as e:

            print(f"Extraction failed: {url} -> {e}")

    return docs

In [13]:
import requests

@tool
def get_weather(query: str) -> list:
    """Search weatherapi to get the current weather."""
    base_url = "http://api.weatherapi.com/v1/current.json"
    complete_url = f"{base_url}?key={WEATHER_API_KEY}&q={query}&aqi=no"
    response = requests.get(complete_url)
    data = response.json()
    if data.get("location"):
        return data
    else:
        return "Weather Data Not Found"

In [14]:
get_weather.invoke("Bangalore")

{'location': {'name': 'Bangalore',
  'region': 'Karnataka',
  'country': 'India',
  'lat': 12.9833,
  'lon': 77.5833,
  'tz_id': 'Asia/Kolkata',
  'localtime_epoch': 1786901484,
  'localtime': '2026-08-16 23:01'},
 'current': {'last_updated_epoch': 1786900500,
  'last_updated': '2026-08-16 22:45',
  'temp_c': 22.9,
  'temp_f': 73.2,
  'is_day': 0,
  'condition': {'text': 'Clear',
   'icon': '//cdn.weatherapi.com/weather/64x64/night/113.png',
   'code': 1000},
  'wind_mph': 13.4,
  'wind_kph': 21.6,
  'wind_degree': 264,
  'wind_dir': 'W',
  'pressure_mb': 1013.0,
  'pressure_in': 29.92,
  'precip_mm': 0.0,
  'precip_in': 0.0,
  'humidity': 66,
  'cloud': 18,
  'feelslike_c': 20.8,
  'feelslike_f': 69.4,
  'windchill_c': 22.9,
  'windchill_f': 73.2,
  'heatindex_c': 24.9,
  'heatindex_f': 76.9,
  'dewpoint_c': 16.1,
  'dewpoint_f': 61.0,
  'vis_km': 10.0,
  'vis_miles': 6.0,
  'uv': 0.0,
  'gust_mph': 18.8,
  'gust_kph': 30.2,
  'will_it_rain': 0,
  'chance_of_rain': 5,
  'will_it_snow'

In [37]:
tools = [multiply, search_web_extract_info, get_weather]

# Now we need to tell LLM that these are the toll you have access.

chatgpt_with_tools = chatgpt.bind_tools(tools)

In [38]:
# We have asked to return the tool calls, not call the tool.
prompt = """
            Given only the tools at your disposal, mention tool calls for the following tasks:
            Do not change the query given for any search tasks
            1. What is 2.1 times 3.5
            2. What is the current weather in "Delhi capital of india" today
            3. What are the 4 major Agentic AI Design Patterns
         """

results = chatgpt_with_tools.invoke(prompt)

In [34]:
results

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 189, 'total_tokens': 265, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e160a45017', 'id': 'chatcmpl-EDZKLiaw5iNOu6Y3wYe7tJ73vRLKf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a00bb4-6727-7743-ad57-d54eafb6a10c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 2.1, 'b': 3.5}, 'id': 'call_LKvtYsMsU6cZXYfuirT82nRV', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'query': 'Delhi capital of india'}, 'id': 'call_2wvw0LCZS1SK3f1445VPdyy6', 'type': 'tool_call'}, {'name': 'search_web_extract_info', 'args': {'query': '4 major Agentic AI Design Patterns'}, 'id':

In [24]:
from IPython.display import display, Markdown

In [39]:
# LLM has suggested to call correct tool with correct arguments.

results.tool_calls

[{'name': 'multiply',
  'args': {'a': 2.1, 'b': 3.5},
  'id': 'call_510Blz9ytFt9DQDMQZ8JS6nQ',
  'type': 'tool_call'},
 {'name': 'get_weather',
  'args': {'query': 'Delhi capital of india'},
  'id': 'call_68oLHY79xDXITQFZhnG4gKnt',
  'type': 'tool_call'},
 {'name': 'search_web_extract_info',
  'args': {'query': '4 major Agentic AI Design Patterns'},
  'id': 'call_mEJ7mROfCckMSP90Fp3CLblp',
  'type': 'tool_call'}]

For now lets call these tool manually, later we will see Agentic AI which will call these tools automatically. lets extract the tool name and arguments and call the tolls manually.

In [40]:
toolkit = {
    "multiply": multiply,
    "search_web_extract_info": search_web_extract_info,
    "get_weather": get_weather
}

for tool_call in results.tool_calls:
    selected_tool = toolkit[tool_call["name"].lower()]
    print(f"Calling tool: {tool_call['name']}")
    tool_output = selected_tool.invoke(tool_call["args"])
    print(tool_output)
    print()

Calling tool: multiply
7.3500000000000005

Calling tool: get_weather
{'location': {'name': 'Delhi', 'region': 'Delhi', 'country': 'India', 'lat': 28.6667, 'lon': 77.2167, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1786902830, 'localtime': '2026-08-16 23:23'}, 'current': {'last_updated_epoch': 1786902300, 'last_updated': '2026-08-16 23:15', 'temp_c': 32.8, 'temp_f': 91.0, 'is_day': 0, 'condition': {'text': 'Patchy rain nearby', 'icon': '//cdn.weatherapi.com/weather/64x64/night/176.png', 'code': 1063}, 'wind_mph': 5.1, 'wind_kph': 8.3, 'wind_degree': 246, 'wind_dir': 'WSW', 'pressure_mb': 1000.0, 'pressure_in': 29.52, 'precip_mm': 0.05, 'precip_in': 0.0, 'humidity': 55, 'cloud': 91, 'feelslike_c': 37.1, 'feelslike_f': 98.8, 'windchill_c': 32.8, 'windchill_f': 91.0, 'heatindex_c': 37.5, 'heatindex_f': 99.6, 'dewpoint_c': 22.4, 'dewpoint_f': 72.3, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 0.0, 'gust_mph': 7.1, 'gust_kph': 11.4, 'will_it_rain': 0, 'chance_of_rain': 14, 'will_it_snow': 0, 'ch

  0%|          | 0/5 [00:00<?, ?it/s]

Processing URL: https://www.youtube.com/watch?v=MrD9tCNpOvU


 20%|██        | 1/5 [00:01<00:06,  1.56s/it]

Processing URL: https://www.tungstenautomation.com/blog/build-enterprise-grade-ai-agents-agentic-design-patterns


 40%|████      | 2/5 [00:03<00:05,  1.89s/it]

... (3 outputs left out)

 80%|████████  | 4/5 [00:05<00:01,  1.22s/it]

Processing URL: https://github.com/nibzard/awesome-agentic-patterns


100%|██████████| 5/5 [00:06<00:00,  1.26s/it]

['Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services\n# YouTube\n\n## Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services\n\n### Video Metadata\n- **Keywords:** video, sharing, camera phone, video phone, free, upload\n- **Runtime:** PT9M54S\n\n### Description\nThe video explores four foundational design patterns in agentic AI systems: reflection, tool use, planning, and multi-agent collaboration. These patterns represent the evolution of AI from basic problem-solving to sophisticated autonomous systems capable of making complex decisions with minimal human oversight. Through practical examples using Amazon Bedrock, the video demonstrates how AI agents can self-evaluate, utilize external tools, break down complex tasks, and collaborate effectively. The presentation emphasizes how these patterns enable AI systems to work independently and cooperatively while adapting to various challenges, ultimately transforming traditional work proce

In [43]:
len(tool_output)  # This will display the output of the third tool call, which is the weather information for Delhi.

5

In [ ]:
from IPython.display import display, Markdown

Tool calling for LLMs without native support for tool or function calling

Some models like ChatGPT have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling.

Here we will explore an alternative method to invoke tools if you're using a model that does not natively support tool calling (even though we use ChatGPT here which supports it, we will assume it could be any LLM which doesn't support tool calling).

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools.

In [44]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate


from langchain_core.tools import render_text_description


#In this case we have to provide all the details abot tools manually as we can not do bind_tool here.
rendered_tools = render_text_description(tools)

print(rendered_tools)

multiply(a: float, b: float) -> float - Use this tool to multiply two numbers.
search_web_extract_info(query: str) -> list - Search the web for a query and extract useful information from search results.
get_weather(query: str) -> list - Search weatherapi to get the current weather.


In [45]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user instructions, for each instruction do the following:
 - Return the name and input of the tool to use.
 - Return your response as a JSON blob with 'name' and 'arguments' keys.
 - The `arguments` should be a dictionary, with keys corresponding
   to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("user", "{input}")
    ]
)

In [46]:
instructions = [
                  {"input" : "What is 2.1 times 3.5"},
                  {"input" : "What is the current weather in Greenland"},
                  {"input" : "Tell me about the current state of Agentic AI in the industry" }
               ]

In [47]:
from langchain_core.output_parsers import JsonOutputParser

chain = (prompt
            |
         chatgpt
            |
         JsonOutputParser())

In [48]:
responses = chain.map().invoke(instructions)

In [49]:
responses

[{'name': 'multiply', 'arguments': {'a': 2.1, 'b': 3.5}},
 {'name': 'get_weather', 'arguments': {'query': 'Greenland'}},
 {'name': 'search_web_extract_info',
  'arguments': {'query': 'current state of Agentic AI in the industry 2023'}}]

In [50]:
toolkit = {
    "multiply": multiply,
    "search_web_extract_info": search_web_extract_info,
    "get_weather": get_weather
}

for tool_call in responses:
    selected_tool = toolkit[tool_call["name"].lower()]
    print(f"Calling tool: {tool_call['name']}")
    tool_output = selected_tool.invoke(tool_call["arguments"])
    print(tool_output)
    print()

Calling tool: multiply
7.3500000000000005

Calling tool: get_weather
{'location': {'name': 'Nuuk', 'region': 'Vestgronland', 'country': 'Greenland', 'lat': 64.183, 'lon': -51.75, 'tz_id': 'America/Nuuk', 'localtime_epoch': 1786903440, 'localtime': '2026-08-16 17:04'}, 'current': {'last_updated_epoch': 1786903200, 'last_updated': '2026-08-16 17:00', 'temp_c': 5.4, 'temp_f': 41.8, 'is_day': 1, 'condition': {'text': 'Sunny', 'icon': '//cdn.weatherapi.com/weather/64x64/day/113.png', 'code': 1000}, 'wind_mph': 18.6, 'wind_kph': 29.9, 'wind_degree': 186, 'wind_dir': 'S', 'pressure_mb': 1012.0, 'pressure_in': 29.87, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 83, 'cloud': 11, 'feelslike_c': 0.6, 'feelslike_f': 33.1, 'windchill_c': 0.6, 'windchill_f': 33.1, 'heatindex_c': 5.4, 'heatindex_f': 41.8, 'dewpoint_c': 2.7, 'dewpoint_f': 36.9, 'vis_km': 10.0, 'vis_miles': 6.0, 'uv': 1.9, 'gust_mph': 30.9, 'gust_kph': 49.7, 'will_it_rain': 0, 'chance_of_rain': 11, 'will_it_snow': 0, 'chance_of_snow

  0%|          | 0/5 [00:00<?, ?it/s]

Processing URL: https://market.us/report/agentic-ai-market


 20%|██        | 1/5 [00:00<00:03,  1.04it/s]

Processing URL: https://dotsolved.com/blog/from-generative-ai-to-agentic-ai-2023-2024-and-beyond


 40%|████      | 2/5 [00:02<00:04,  1.57s/it]

... (3 outputs left out)

 80%|████████  | 4/5 [00:04<00:00,  1.17it/s]

Processing URL: https://svitla.com/blog/agentic-ai-market-trends-2026


100%|██████████| 5/5 [00:06<00:00,  1.30s/it]

['Agentic AI Market Size, Share, Trends | CAGR of 43.8%\n[![One Stop Shop For Reports](/wp-content/themes/marketus/images/logo-dark.svg)\n![One Stop Shop For Reports](/wp-content/themes/marketus/images/logo-white.svg)](/)\n\nBy Name\nBy Code\n\n* [All Reports](/reports/)\n* [All Sectors](/categories/)\n  + [Chemicals & Materials](https://market.us/reports/chemical-and-materials/)\n    - [Advanced Materials](https://market.us/reports/advanced-materials/)\n    - [Bulk Chemicals](https://market.us/reports/bulk-chemicals/)\n    - [Coatings | Paints and Additives](https://market.us/reports/coatings-paints-and-additives/)\n    - [Composites](https://market.us/reports/composites/)\n    - [Renewable | Speciality chemicals](https://market.us/reports/renewable-speciality-chemicals/)\n  + [Consumer Goods](https://market.us/reports/consumer-goods/)\n    - [Baby Products](https://market.us/reports/baby-products/)\n    - [Consumer Electronics](https://market.us/reports/consumer-electronics/)\n    - 

In [51]:
for doc in tool_output:
    print(doc)
    print()

Agentic AI Market Size, Share, Trends | CAGR of 43.8%
[![One Stop Shop For Reports](/wp-content/themes/marketus/images/logo-dark.svg)
![One Stop Shop For Reports](/wp-content/themes/marketus/images/logo-white.svg)](/)

By Name
By Code

* [All Reports](/reports/)
* [All Sectors](/categories/)
  + [Chemicals & Materials](https://market.us/reports/chemical-and-materials/)
    - [Advanced Materials](https://market.us/reports/advanced-materials/)
    - [Bulk Chemicals](https://market.us/reports/bulk-chemicals/)
    - [Coatings | Paints and Additives](https://market.us/reports/coatings-paints-and-additives/)
    - [Composites](https://market.us/reports/composites/)
    - [Renewable | Speciality chemicals](https://market.us/reports/renewable-speciality-chemicals/)
  + [Consumer Goods](https://market.us/reports/consumer-goods/)
    - [Baby Products](https://market.us/reports/baby-products/)
    - [Consumer Electronics](https://market.us/reports/consumer-electronics/)
    - [Consumer Packaging]

Test Tool Calling with LLM